In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tqdm.keras import TqdmCallback

# ---- Keras-serializable preprocess layer (Keras 3 safe) ----
@keras.utils.register_keras_serializable(package="custom")
class EffNetV2Preprocess(layers.Layer):
    def call(self, inputs):
        inputs = tf.cast(inputs, tf.float32)
        # IMPORTANT: your pipeline provides [0,1] (because you divided by 255 earlier),
        # but preprocess_input expects [0,255]. So convert here:
        return preprocess_input(inputs * 255.0)

    def get_config(self):
        return super().get_config()

def build_classifier_efficientnetv2(
    input_size=224,
    dropout=0.3,
    wd=1e-4,
    weights="imagenet",
    freeze_backbone=True
):
    inp = keras.Input(shape=(CFG.img_h, CFG.img_w, CFG.img_c), name="image")

    x = layers.Resizing(input_size, input_size, name="resize")(inp)
    x = EffNetV2Preprocess(name="effnetv2_preprocess")(x)

    backbone = EfficientNetV2S(
        include_top=False,
        weights=weights,
        input_shape=(input_size, input_size, 3),
        pooling="avg",
    )
    backbone.trainable = not freeze_backbone

    # BN stability when frozen
    feats = backbone(x, training=False if freeze_backbone else True)

    h = layers.Dense(
        256, activation="relu",
        kernel_regularizer=keras.regularizers.l2(wd),
        name="head_dense",
    )(feats)
    h = layers.Dropout(dropout, name="head_dropout")(h)
    out = layers.Dense(CFG.num_classes, name="logits")(h)

    model = keras.Model(inp, out, name="effnetv2s_classifier")
    return model, backbone

def compile_classifier(m, lr=1e-3):
    m.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

# --- quick sanity check: make sure labels/ranges look right ---
xb, yb = next(iter(train_ds))
print("sanity x range:", float(tf.reduce_min(xb)), float(tf.reduce_max(xb)))  # should be ~0..1
print("sanity y shape/dtype:", yb.shape, yb.dtype)
print("sanity y min/max:", int(tf.reduce_min(yb)), int(tf.reduce_max(yb)))   # should be 0..9

# --- Try to load saved model first ---
clf = None
if hasattr(CFG, "clf_path") and CFG.clf_path and os.path.exists(CFG.clf_path):
    try:
        clf = keras.models.load_model(CFG.clf_path)
        print("Loaded saved classifier:", CFG.clf_path)
    except Exception as e:
        print("Failed to load saved classifier. Will retrain. Error:", repr(e))
        clf = None
else:
    print("No saved classifier found at:", getattr(CFG, "clf_path", None))

# --- If not loaded, train and save ---
if clf is None:
    clf, backbone = build_classifier_efficientnetv2(
        input_size=224,
        dropout=0.3,
        wd=1e-4,
        weights="imagenet",
        freeze_backbone=True,
    )
    compile_classifier(clf, lr=1e-3)

    clf.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG.clf_epochs,
        callbacks=[TqdmCallback(verbose=1)],
        verbose=0,
    )

    # Optional fine-tuning
    if getattr(CFG, "clf_finetune_epochs", 0) and CFG.clf_finetune_epochs > 0:
        backbone.trainable = True
        compile_classifier(clf, lr=1e-4)
        clf.fit(
            train_ds,
            validation_data=val_ds,
            epochs=CFG.clf_epochs + CFG.clf_finetune_epochs,
            initial_epoch=CFG.clf_epochs,
            callbacks=[TqdmCallback(verbose=1)],
            verbose=0,
        )

    clf.save(CFG.clf_path)
    print("Saved classifier:", CFG.clf_path)

# Ensure model is compiled
try:
    compile_classifier(clf, lr=1e-3)
except Exception:
    pass

test_acc = clf.evaluate(test_ds, verbose=0)[1]
print("Classifier test acc:", test_acc)
